In [1]:
import polars as pl
import polars.selectors as cs
import wandb
import numpy as np
import plotly.express as px
import matplotlib.pyplot as plt

api = wandb.Api()
artifact = api.artifact("flood-forecasting/flood-dataset:latest")
artifact_dir = artifact.download()

df = (
    pl.scan_parquet(f"{artifact_dir}/flood_model.parquet")
    .limit(100_000)
    .filter(pl.col("observation_hour") < pl.lit("2024-01-01").str.to_datetime())
    .collect()
)

print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from C:\Users\sacha\_netrc.
wandb: Downloading large artifact 'flood-dataset:latest', 4549.24MB. 1 files...
wandb:   1 of 1 files downloaded.  
Done. 00:00:00.5 (8316.7MB/s)


Loaded: 91,039 rows x 460 columns


In [2]:
numeric_df = df.select(cs.numeric()).to_pandas()

target_corr = (
    numeric_df.corr()[["streamflow_cfs_max"]]
    .drop(index="streamflow_cfs_max")
    .sort_values("streamflow_cfs_max", ascending=False)
)

low_corr = target_corr[target_corr["streamflow_cfs_max"].abs() < 0.1]

print(f"Total features: {len(target_corr)}")
print(f"Low correlation with target (<0.1): {len(low_corr)}")
print("\nTop 15 correlated:")
print(target_corr.head(15))
print("\nBottom 15 correlated:")
print(target_corr.tail(15))

Total features: 446
Low correlation with target (<0.1): 144

Top 15 correlated:
                              streamflow_cfs_max
streamflow_cfs_mean                     0.999985
streamflow_cfs_min                      0.999943
streamflow_cfs_target_1h                0.999933
rip800_12                               0.495939
snow_ice_nlcd06                         0.495922
cdl_durum_wheat                         0.494405
padcat1_pct_basin                       0.476530
rip100_12                               0.473063
artificial_path_mainstem_pct            0.466084
wet_pc_ug2                              0.451685
padcat2_pct_basin                       0.444597
barren_nlcd06                           0.404235
wet_pc_ug1                              0.402321
topwet                                  0.402097
hga                                     0.389165

Bottom 15 correlated:
                       streamflow_cfs_max
cdl_wwht_soy_dbl_crop                 NaN
cdl_pasture_grass            

In [3]:
# drop NaN correlation or absolute correlation < 0.1
keep_always = [
    "streamflow_cfs_max", "streamflow_cfs_mean", "streamflow_cfs_min",
    "DRAIN_SQKM",
    "longitude",
    "latitude"
]

drop_cols = target_corr[
    (target_corr["streamflow_cfs_max"].abs() < 0.2) |
    (target_corr["streamflow_cfs_max"].isna())
].index.tolist()

force_drop = [
    "pnv_pc_u10",       
    "mains800_52",      
    "rip800_12",        
    "streamflow_cfs_target_1h", 
    "gage_height_ft_target_1h"

]

drop_cols = [c for c in drop_cols if c not in keep_always]
drop_cols += [c for c in force_drop if c in numeric_df.columns and c not in keep_always]

drop_cols = [c for c in drop_cols if c not in keep_always]

reduced_df = numeric_df.drop(columns=drop_cols)

print(f"Dropped: {len(drop_cols)} columns")
print(f"Remaining: {reduced_df.shape[1]} columns")

Dropped: 338 columns
Remaining: 110 columns


In [4]:
corr_reduced = reduced_df.corr()

fig = px.imshow(
    corr_reduced,
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    title="Correlation Matrix: High-Signal Features (|r| > 0.2 with target)",
    width=1200, height=1200
)

fig.update_layout(coloraxis_colorbar=dict(title="r"))
fig.show()

In [5]:
corr_matrix = reduced_df.corr().abs()

# Find pairs with r > 0.9 
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_pairs = (
    upper.stack()
    .reset_index()
    .rename(columns={"level_0": "feature_1", "level_1": "feature_2", 0: "r"})
    .query("r > 0.9")
    .sort_values("r", ascending=False)
)

print(f"Hyper-correlated pairs (r > 0.9): {len(high_corr_pairs)}")
print(high_corr_pairs.to_string())

Hyper-correlated pairs (r > 0.9): 815
                         feature_1                     feature_2         r
4564                    tbi_cl_smj                    tec_cl_smj  1.000000
113                      longitude                   longitude_1  1.000000
2677                    tmp_dc_smx                    tmp_dc_s07  1.000000
217            streamflow_cfs_mean            streamflow_cfs_max  0.999985
218            streamflow_cfs_mean            streamflow_cfs_min  0.999984
742                     DRAIN_SQKM            nwis_drainage_area  0.999973
324             streamflow_cfs_max            streamflow_cfs_min  0.999943
1664                        perdun                    tec_cl_smj  0.999906
1663                        perdun                    tbi_cl_smj  0.999906
2835                    tmp_dc_s05                    tmp_dc_s06  0.999341
3370                    aet_mm_syr                    aet_mm_s07  0.999148
2600                    tmp_dc_syr                    tmp_dc_s

In [6]:
from collections import Counter

# Count how many times each feature appears in a pair
all_features = high_corr_pairs["feature_1"].tolist() + high_corr_pairs["feature_2"].tolist()
freq = Counter(all_features)

to_drop = set()
for _, row in high_corr_pairs.iterrows():
    f1, f2 = row["feature_1"], row["feature_2"]
    if f1 in to_drop or f2 in to_drop:
        continue
    # Drop whichever appears in more pairs 
    drop = f1 if freq[f1] >= freq[f2] else f2
    if drop not in keep_always:
        to_drop.add(drop)

final_df = reduced_df.drop(columns=list(to_drop))

print(f"Dropped: {len(to_drop)} hyper-correlated columns")
print(f"Final feature count: {final_df.shape[1]}")
print(f"Kept: {list(final_df.columns)}")

Dropped: 91 hyper-correlated columns
Final feature count: 19
Kept: ['latitude', 'longitude', 'streamflow_cfs_mean', 'streamflow_cfs_max', 'streamflow_cfs_min', 'specific_humidity_kgkg', 'DRAIN_SQKM', 'strahler_max', 'artificial_path_pct', 'wb5100_ann_mm', 'snw_pc_syr', 'snow_ice_nlcd06', 'barren_nlcd06', 'mains100_plant', 'hga', 'hgc', 'bulk_density_avg', 'elev_max_m', 'aspect_deg']


In [7]:
# Correlation with target
final_target_corr = (
    final_df.corr()[["streamflow_cfs_max"]]
    .drop(index="streamflow_cfs_max")  
    .sort_values("streamflow_cfs_max", ascending=False)
    .round(3)
)
print("Correlation with streamflow_cfs_max:")
print(final_target_corr.to_string())

Correlation with streamflow_cfs_max:
                        streamflow_cfs_max
streamflow_cfs_mean                  1.000
streamflow_cfs_min                   1.000
snow_ice_nlcd06                      0.496
barren_nlcd06                        0.404
hga                                  0.389
hgc                                  0.347
bulk_density_avg                     0.316
artificial_path_pct                  0.291
specific_humidity_kgkg               0.238
snw_pc_syr                           0.226
elev_max_m                           0.212
DRAIN_SQKM                           0.205
wb5100_ann_mm                        0.204
strahler_max                         0.201
latitude                            -0.009
aspect_deg                          -0.214
mains100_plant                      -0.270
longitude                           -0.410


In [8]:
SITE_INFO = [
    "site_id", "observation_hour", "station_name"
]

STATIC_FEATURES = [
    "longitude", "latitude", "DRAIN_SQKM", "artificial_path_pct",
    "wb5100_ann_mm", "snw_pc_syr", "snow_ice_nlcd06", "barren_nlcd06",
    "mains100_plant", "hga", "hgc", "bulk_density_avg", "elev_max_m", "aspect_deg", "strahler_max"
]

DYNAMIC_FEATURES = [
    "streamflow_cfs_mean", "streamflow_cfs_max", "streamflow_cfs_min",
    "gage_height_ft_mean",
    "precipitation_mm",
    "temperature_c",
    "potential_evaporation_mm",
    "specific_humidity_kgkg",
    "shortwave_radiation_wm2",
    "longwave_radiation_wm2",
    "wind_speed_ms",
    "surface_pressure_pa",
    "cape_jkg",
    "convective_precip_fraction",
]

TARGET = "streamflow_cfs_mean" 

print(f"Static features: {len(STATIC_FEATURES)}")
print(f"Dynamic features: {len(DYNAMIC_FEATURES)}")
print(f"Target: {TARGET}")

Static features: 15
Dynamic features: 14
Target: streamflow_cfs_mean


In [9]:
# Create lagged target for correlation analysis
lag_hours = 24

lag_corr_df = (
    df.select(DYNAMIC_FEATURES + ["observation_hour", "site_id"])
    .sort(["site_id", "observation_hour"])
    .with_columns(
        pl.col("streamflow_cfs_mean")
        .shift(-lag_hours)
        .over("site_id")
        .alias("target_24h")
    )
    .drop_nulls("target_24h")
    .to_pandas()
)

dynamic_corr = (
    lag_corr_df[DYNAMIC_FEATURES]
    .corrwith(lag_corr_df["target_24h"])
    .sort_values(ascending=False)
    .round(3)
)

print(f"Correlation of dynamic features with streamflow 24h ahead:")
print(dynamic_corr.to_string())

Correlation of dynamic features with streamflow 24h ahead:
streamflow_cfs_max            0.994
streamflow_cfs_mean           0.994
streamflow_cfs_min            0.994
specific_humidity_kgkg        0.238
temperature_c                 0.172
longwave_radiation_wm2        0.159
gage_height_ft_mean           0.129
surface_pressure_pa           0.100
convective_precip_fraction    0.088
shortwave_radiation_wm2       0.084
potential_evaporation_mm      0.083
precipitation_mm              0.052
cape_jkg                      0.046
wind_speed_ms                -0.038


In [10]:
# Static correlation matrix
static_cols = [c for c in STATIC_FEATURES]
corr_static = df[static_cols].to_pandas().corr()

fig_static = px.imshow(
    corr_static,
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    title=f"Static Feature Correlation Matrix ({len(static_cols)} features)",
    width=700, height=700,
    text_auto=".2f"
)
fig_static.update_layout(coloraxis_colorbar=dict(title="r"))
fig_static.show()

# Dynamic correlation matrix
dynamic_cols = [c for c in DYNAMIC_FEATURES]
corr_dynamic = df[dynamic_cols].to_pandas().corr()

fig_dynamic = px.imshow(
    corr_dynamic,
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    title=f"Dynamic Feature Correlation Matrix ({len(dynamic_cols)} features)",
    width=700, height=700,
    text_auto=".2f"
)
fig_dynamic.update_layout(coloraxis_colorbar=dict(title="r"))
fig_dynamic.show()




## Selected Features

Starting from 460 columns, we reduced to 14 static and 14 dynamic features using correlation analysis on ~91k rows (2007–2023).

**Process:** dropped features with |r| < 0.2 against streamflow_cfs_max, then dropped hyper-correlated pairs (|r| > 0.9) using a greedy algorithm. `DRAIN_SQKM` and `longitude` were force-kept due to physical importance.

**Target:** `streamflow_cfs_mean` at 24h ahead, computed in preprocessing.

Dynamic features do not have an r because they were added after the analysis. This is because they need to be lagged to show influence. 

| Feature | Category | Description | r |
|---|---|---|---|
| `snow_ice_nlcd06` | Snow | Snow/ice land cover % | +0.50 |
| `snw_pc_syr` | Snow | Annual snow cover % | +0.23 |
| `barren_nlcd06` | Land Cover | Bare land %, high runoff areas | +0.41 |
| `mains100_plant` | Land Cover | Vegetation % along mainstem, dampens peaks | -0.27 |
| `hga` | Soils | Soil group A % (high infiltration) | +0.39 |
| `hgc` | Soils | Soil group C % (high runoff) | +0.35 |
| `bulk_density_avg` | Soils | Soil compactness, affects infiltration | +0.32 |
| `artificial_path_pct` | Hydrology | % artificial stream network (canals etc.) | +0.29 |
| `wb5100_ann_mm` | Hydrology | Annual water balance (precip minus ET) | +0.20 |
| `DRAIN_SQKM` | Basin | Drainage area (sq km) | +0.21 |
| `strahler_max` | Basin | Max Strahler stream order, proxy for network complexity | +0.21 |
| `elev_max_m` | Topography | Max basin elevation, controls snowmelt | +0.21 |
| `aspect_deg` | Topography | Slope direction, affects snowmelt | -0.21 |
| `longitude` | Spatial | East-west position, proxy for aridity | -0.41 |
| `latitude` | Spatial | North-south position | -0.01 |
| `streamflow_cfs_mean` | Dynamic | Mean hourly streamflow (CFS) | +0.994 |
| `streamflow_cfs_max` | Dynamic | Max hourly streamflow (CFS) | +0.994 |
| `streamflow_cfs_min` | Dynamic | Min hourly streamflow (CFS) | +0.994 |
| `specific_humidity_kgkg` | Dynamic | Atmospheric moisture (kg/kg) | +0.238 |
| `temperature_c` | Dynamic | Air temperature (°C) | +0.172 |
| `longwave_radiation_wm2` | Dynamic | Longwave radiation (W/m²) | +0.159 |
| `gage_height_ft_mean` | Dynamic | Mean gage height (ft) | +0.129 |
| `surface_pressure_pa` | Dynamic | Surface pressure (Pa) | +0.100 |
| `convective_precip_fraction` | Dynamic | Fraction of convective precipitation | +0.088 |
| `shortwave_radiation_wm2` | Dynamic | Shortwave radiation (W/m²) | +0.084 |
| `potential_evaporation_mm` | Dynamic | Potential evaporation (mm) | +0.083 |
| `precipitation_mm` | Dynamic | Precipitation (mm) | +0.052 |
| `cape_jkg` | Dynamic | Convective instability (J/kg) | +0.046 |
| `wind_speed_ms` | Dynamic | Wind speed (m/s) | -0.038 |